# RECAP Ablation Study — Conditioning Signal Variations

**Two ablations that change what the advantage label means:**

| Ablation | What changes | Conditions |
|----------|-------------|------------|
| **A1: Label Granularity** | How frames get labeled | (a) Value-head thresholded (b) Continuous value embedding |
| **A2: Data Subset** | Which data the model sees | (a) Negative-only (b) Positive-only |

The binary episode-level baseline (A1-baseline / A2-baseline) is the run from `recap_final.ipynb` — we don't re-run it.

**Architecture:** Suffix-time injection (same as `recap_final.ipynb`).
Advantage embedding added to the first suffix token inside `embed_suffix`,
rebuilt every denoising step, KV cache safe.

**Value head:** Trained in `hrishikeshs-notebook-apr-23.ipynb` (AUC 0.878).
Loaded from `/mnt/rollouts/vlm_features/value_head.pt`.

**Compute:** 4 training runs × ~25-30 min each ≈ 2 hours total.

## Step 1 — Environment

In [5]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
DRIVE_ROOT = "/mnt/rollouts"
os.environ["HF_HOME"]         = f"{DRIVE_ROOT}/hf_cache"
os.environ["HF_LEROBOT_HOME"] = f"{DRIVE_ROOT}/lerobot_cache"

In [1]:
!apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1
!pip install -q "lerobot[smolvla,libero]" wandb scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
print(f"CUDA: {torch.cuda.is_available()}  |  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

CUDA: True  |  NVIDIA A100-SXM4-40GB


## Step 2 — Constants

In [3]:
import wandb
import numpy as np
from pathlib import Path

WANDB_PROJECT = "IDL_34_RECAP"
WANDB_ENTITY  = "idl_34"
WANDB_MODE    = "online"

DRIVE_ROOT = "/mnt/rollouts"
ROLLOUT_DATASET_DIR = Path(DRIVE_ROOT) / "rollouts_with_labels"
FEATURE_CACHE       = Path(DRIVE_ROOT) / "vlm_features"
ABLATION_ROOT       = Path(DRIVE_ROOT) / "recap_ablations"
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)

BC_POLICY_REPO = "HuggingFaceVLA/smolvla_libero"
LIBERO_SUITE   = "libero_spatial"
ADV_NEG, ADV_POS, ADV_NULL = 0, 1, 2

TASK_DESCRIPTIONS = {
    0: "pick up the black bowl between the plate and the ramekin and place it on the plate",
    1: "pick up the black bowl from table center and place it on the plate",
    2: "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    3: "pick up the black bowl next to the cookie box and place it on the plate",
    4: "pick up the black bowl next to the plate and place it on the plate",
    5: "pick up the black bowl next to the ramekin and place it on the plate",
    6: "pick up the black bowl on the cookie box and place it on the plate",
    7: "pick up the black bowl on the ramekin and place it on the plate",
    8: "pick up the black bowl on the stove and place it on the plate",
    9: "pick up the black bowl on the wooden cabinet and place it on the plate",
}

device = torch.device("cuda")
print(f"Rollout data: {ROLLOUT_DATASET_DIR}")
print(f"Feature cache: {FEATURE_CACHE}")
print(f"Ablation output: {ABLATION_ROOT}")

Rollout data: /mnt/rollouts/rollouts_with_labels
Feature cache: /mnt/rollouts/vlm_features
Ablation output: /mnt/rollouts/recap_ablations


## Step 3 — LIBERO config

In [4]:
import yaml, libero
libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(parents=True, exist_ok=True)
libero_data_root = Path(DRIVE_ROOT) / "libero_data"
libero_data_root.mkdir(parents=True, exist_ok=True)
pkg_dir = Path(libero.__file__).parent / "libero"
with open(libero_config_dir / "config.yaml", "w") as f:
    yaml.safe_dump({
        "benchmark_root": str(pkg_dir), "bddl_files": str(pkg_dir / "bddl_files"),
        "init_states": str(pkg_dir / "init_files"), "datasets": str(libero_data_root),
        "assets": str(pkg_dir / "assets"),
    }, f)
print("LIBERO config written.")

LIBERO config written.


## Step 4 — Advantage modules (discrete + continuous)

Two advantage injection modes:
1. **Discrete (3-way embedding):** same as recap_final — used for binary, thresholded, neg-only, pos-only
2. **Continuous MLP:** takes a scalar V(s) ∈ [0,1] and projects it to expert_hidden_size

In [5]:
import torch.nn as nn
import torch.nn.functional as F


class AdvantageEmbedding(nn.Module):
    """Discrete 3-way advantage embedding (same as recap_final).
    Slots: A_neg=0, A_pos=1, A_null=2 (zero-init)."""
    def __init__(self, expert_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(3, expert_hidden_size)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.embedding.weight[ADV_NULL].zero_()

    def forward(self, labels):
        return self.embedding(labels)  # (B, expert_hidden)


class ContinuousAdvantageEmbedding(nn.Module):
    """Projects a continuous scalar V(s) in [0,1] to expert_hidden_size.
    Small MLP: scalar -> 64 -> expert_hidden_size.
    Input is centered around 0.5 so V=0.5 produces near-zero output."""
    def __init__(self, expert_hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.GELU(),
            nn.Linear(64, expert_hidden_size),
        )
        # Init last layer small so initial output is near-zero
        nn.init.normal_(self.net[2].weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.net[2].bias)

    def forward(self, values):
        """values: (B,) float tensor in [0,1]. Returns (B, expert_hidden)."""
        x = (values.float() - 0.5).unsqueeze(-1)  # (B, 1)
        return self.net(x)  # (B, expert_hidden)


def _patched_embed_suffix_discrete(self, noisy_actions, timestep):
    """Patched embed_suffix: discrete advantage added to first suffix token."""
    embs, pad_masks, att_masks = self.__class__._orig_embed_suffix(self, noisy_actions, timestep)
    label = getattr(self, "_train_advantage_label",
                    getattr(self, "_eval_advantage_label", ADV_POS))
    bs = embs.shape[0]
    if isinstance(label, torch.Tensor):
        adv_labels = label.to(embs.device)
    else:
        adv_labels = torch.full((bs,), int(label), dtype=torch.long, device=embs.device)
    adv_emb = self.advantage_token(adv_labels)  # (B, expert_hidden)
    new_embs = embs.clone()
    new_embs[:, 0, :] = new_embs[:, 0, :] + adv_emb.to(embs.dtype)
    return new_embs, pad_masks, att_masks


def _patched_embed_suffix_continuous(self, noisy_actions, timestep):
    """Patched embed_suffix: continuous advantage value added to first suffix token."""
    embs, pad_masks, att_masks = self.__class__._orig_embed_suffix(self, noisy_actions, timestep)
    values = getattr(self, "_train_advantage_values",
                     getattr(self, "_eval_advantage_value", None))
    bs = embs.shape[0]
    if values is None:
        values = torch.ones(bs, device=embs.device)  # default: V=1.0 at inference
    elif isinstance(values, (int, float)):
        values = torch.full((bs,), float(values), device=embs.device)
    else:
        values = values.to(embs.device).float()
    adv_emb = self.advantage_token_continuous(values)  # (B, expert_hidden)
    new_embs = embs.clone()
    new_embs[:, 0, :] = new_embs[:, 0, :] + adv_emb.to(embs.dtype)
    return new_embs, pad_masks, att_masks


def set_eval_advantage(pol, label):
    """Set discrete advantage label for inference."""
    for attr in ("_train_advantage_label", "_train_advantage_values", "_eval_advantage_value"):
        if hasattr(pol.model, attr): delattr(pol.model, attr)
    pol.model._eval_advantage_label = int(label)


def set_eval_advantage_continuous(pol, value):
    """Set continuous advantage value for inference."""
    for attr in ("_train_advantage_label", "_train_advantage_values", "_eval_advantage_label"):
        if hasattr(pol.model, attr): delattr(pol.model, attr)
    pol.model._eval_advantage_value = float(value)


print("Advantage modules defined: AdvantageEmbedding (discrete), ContinuousAdvantageEmbedding, patches.")

Advantage modules defined: AdvantageEmbedding (discrete), ContinuousAdvantageEmbedding, patches.


## Step 5 — Load value head + compute per-frame V(s) scores

The value head was trained in the apr-23 notebook. We load it and score
every frame in the rollout dataset to get V(s_t) ∈ [0,1] per frame.
These scores are used by A1 conditions (b) and (c).

In [6]:
import torch.nn as nn

class ValueHead(nn.Module):
    def __init__(self, d_in, d_hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_hidden, d_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_hidden, 1),
        )
    def forward(self, x):
        return torch.sigmoid(self.net(x)).squeeze(-1)


# Load cached VLM features and value head checkpoint
feats     = np.load(FEATURE_CACHE / "features.npy", mmap_mode="r")
feat_mean = np.load(FEATURE_CACHE / "feat_mean.npy")
feat_std  = np.load(FEATURE_CACHE / "feat_std.npy")

D = feats.shape[1]
vh = ValueHead(D).to(device)
vh.load_state_dict(torch.load(FEATURE_CACHE / "value_head.pt", map_location=device))
vh.eval()
print(f"Value head loaded. Input dim={D}")

# Score ALL rollout frames in batches
X_all = torch.from_numpy((np.asarray(feats) - feat_mean) / feat_std).float()
SCORE_BATCH = 512
all_scores = []
with torch.no_grad():
    for i in range(0, len(X_all), SCORE_BATCH):
        batch = X_all[i:i+SCORE_BATCH].to(device)
        all_scores.append(vh(batch).cpu())
value_scores = torch.cat(all_scores).numpy()  # (N,) float32 in [0,1]

print(f"Scored {len(value_scores)} frames")
print(f"  V(s) mean={value_scores.mean():.3f}  std={value_scores.std():.3f}")
print(f"  V(s) > 0.5: {(value_scores > 0.5).sum()} frames")
print(f"  V(s) <= 0.5: {(value_scores <= 0.5).sum()} frames")

np.save(ABLATION_ROOT / "value_scores_per_frame.npy", value_scores)

del vh, X_all
torch.cuda.empty_cache()

Value head loaded. Input dim=960
Scored 15462 frames
  V(s) mean=0.211  std=0.270
  V(s) > 0.5: 2624 frames
  V(s) <= 0.5: 12838 frames


## Step 6 — Rollout dataset (with per-frame value scores)

In [7]:
import glob, pandas as pd
from torch.utils.data import Dataset, ConcatDataset, WeightedRandomSampler


def _unpack(x):
    out = []
    stack = [x]
    while stack:
        item = stack.pop()
        if isinstance(item, (list, tuple, np.ndarray)):
            stack.extend(reversed(list(item)) if hasattr(item, "__iter__") else [item])
        else:
            out.append(item)
    return out


class RolloutFrameDataset(Dataset):
    """Rollout dataset with optional per-frame value scores and label filtering."""
    def __init__(self, root, value_scores=None, label_filter=None):
        self.files = sorted(glob.glob(str(Path(root) / "data" / "chunk-000" / "file-*.parquet")))
        assert self.files, f"No parquet files found in {root}/data/chunk-000/"
        self._labels, self._file_idx, self._row_idx, self._task_idx = [], [], [], []
        self._global_idx = []  # maps dataset index -> original global frame index
        tasks_pq = Path(root) / "meta" / "tasks.parquet"
        tdf = pd.read_parquet(tasks_pq)
        self._tasks_map = dict(zip(tdf["task_index"].tolist(), tdf.index.tolist()))
        global_frame = 0
        for fi, f in enumerate(self.files):
            df = pd.read_parquet(f, columns=["advantage_label", "task_index"])
            for r in range(len(df)):
                lbl_raw = df["advantage_label"].iat[r]
                lbl = int(lbl_raw[0]) if hasattr(lbl_raw, "__len__") else int(lbl_raw)
                if label_filter is not None and lbl != label_filter:
                    global_frame += 1
                    continue
                ti_raw = df["task_index"].iat[r]
                ti = int(ti_raw[0]) if hasattr(ti_raw, "__len__") else int(ti_raw)
                self._labels.append(lbl)
                self._task_idx.append(ti)
                self._file_idx.append(fi)
                self._row_idx.append(r)
                self._global_idx.append(global_frame)
                global_frame += 1
        self._labels = np.array(self._labels, dtype=np.int64)
        self._task_idx = np.array(self._task_idx, dtype=np.int64)
        self._global_idx = np.array(self._global_idx, dtype=np.int64)
        self._value_scores = value_scores  # full array indexed by global_idx
        self._cache = {}
        print(f"Rollout frames: {len(self._labels)}  (filter={label_filter})")

    def __len__(self):
        return len(self._labels)

    def _load_file(self, fi):
        if fi in self._cache:
            return self._cache[fi]
        df = pd.read_parquet(self.files[fi],
                             columns=["observation.images.image", "observation.images.image2",
                                      "observation.state", "action"])
        if len(self._cache) >= 4:
            self._cache.pop(next(iter(self._cache)))
        self._cache[fi] = df
        return df

    def __getitem__(self, i):
        df = self._load_file(self._file_idx[i])
        r = self._row_idx[i]
        img1   = np.asarray(_unpack(df["observation.images.image"].iat[r]),  dtype=np.float32).reshape(3, 256, 256)
        img2   = np.asarray(_unpack(df["observation.images.image2"].iat[r]), dtype=np.float32).reshape(3, 256, 256)
        state  = np.asarray(_unpack(df["observation.state"].iat[r]), dtype=np.float32)
        action = np.asarray(_unpack(df["action"].iat[r]), dtype=np.float32)
        v_score = float(self._value_scores[self._global_idx[i]]) if self._value_scores is not None else 0.0
        return {
            "observation.images.image":  torch.from_numpy(img1),
            "observation.images.image2": torch.from_numpy(img2),
            "observation.state":         torch.from_numpy(state),
            "action":                    torch.from_numpy(action),
            "advantage_label":           int(self._labels[i]),
            "value_score":               v_score,
            "task":                      self._tasks_map.get(int(self._task_idx[i]), ""),
        }


# Load full rollout dataset with value scores
full_ds = RolloutFrameDataset(ROLLOUT_DATASET_DIR, value_scores=value_scores, label_filter=None)
print(f"Full dataset: {len(full_ds)} frames  pos={int((full_ds._labels == 1).sum())}  neg={int((full_ds._labels == 0).sum())}")

Rollout frames: 15462  (filter=None)
Full dataset: 15462 frames  pos=7875  neg=7587


## Step 7 — Expert dataset

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("Loading expert dataset ...")
expert_ds_raw = LeRobotDataset("HuggingFaceVLA/libero")

SPATIAL_TASK_STRINGS = set(TASK_DESCRIPTIONS.values())
spatial_episodes = []
for ep_i in range(expert_ds_raw.meta.total_episodes):
    ep = expert_ds_raw.meta.episodes[ep_i]
    if len(ep["tasks"]) == 1 and ep["tasks"][0] in SPATIAL_TASK_STRINGS:
        spatial_episodes.append((ep_i, ep["dataset_from_index"], ep["dataset_to_index"], ep["tasks"][0]))


class ExpertSpatialDataset(Dataset):
    def __init__(self, ds, spatial_episodes):
        self.ds = ds
        self.frame_map = []
        for _, src, dst, task_str in spatial_episodes:
            for f in range(src, dst):
                self.frame_map.append((f, task_str))

    def __len__(self):
        return len(self.frame_map)

    def __getitem__(self, i):
        ds_idx, task_str = self.frame_map[i]
        s = self.ds[ds_idx]
        return {
            "observation.images.image":  s["observation.images.image"],
            "observation.images.image2": s["observation.images.image2"],
            "observation.state":         s["observation.state"],
            "action":                    s["action"],
            "advantage_label":           ADV_POS,
            "value_score":               1.0,  # expert = perfect value
            "task":                      task_str,
        }


expert_ds = ExpertSpatialDataset(expert_ds_raw, spatial_episodes)
print(f"Expert Spatial: {len(expert_ds)} frames, {len(spatial_episodes)} episodes")

## Step 8 — Shared training infrastructure

Collate, loss function, training loop — identical to `recap_final` except
parameterized by which advantage signal to use.

In [9]:
import json, time
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from lerobot.utils.constants import ACTION
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, make_att_2d_masks


def collate(bl):
    return {
        "observation.images.image":  torch.stack([b["observation.images.image"] for b in bl]),
        "observation.images.image2": torch.stack([b["observation.images.image2"] for b in bl]),
        "observation.state":         torch.stack([b["observation.state"] for b in bl]),
        "action":                    torch.stack([b["action"] for b in bl]),
        "advantage_label":           torch.tensor([int(b["advantage_label"]) for b in bl], dtype=torch.long),
        "value_score":               torch.tensor([b["value_score"] for b in bl], dtype=torch.float32),
        "task":                      [b["task"] for b in bl],
    }


def batch_to_device(batch, dev):
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.to(dev, non_blocking=True)
    return batch


def recap_loss(policy, batch, tokenizer, CHUNK_SIZE, MAX_ACTION_DIM,
               adv_labels=None, adv_values=None):
    """Flow-matching loss with advantage conditioning.
    Exactly one of adv_labels (discrete) or adv_values (continuous) must be set."""
    bs  = batch[ACTION].shape[0]
    dev = batch[ACTION].device
    images, img_masks = policy.prepare_images(batch)
    state   = policy.prepare_state(batch)
    actions = policy.prepare_action(batch)
    tok = tokenizer(list(batch["task"]), padding="max_length",
                    max_length=policy.config.tokenizer_max_length,
                    truncation=True, return_tensors="pt")
    lang_tokens = tok["input_ids"].to(dev)
    lang_masks  = tok["attention_mask"].to(dev).bool()
    if actions.ndim == 2:
        actions = actions.unsqueeze(1).expand(-1, CHUNK_SIZE, -1)
    if actions.shape[-1] < MAX_ACTION_DIM:
        actions = F.pad(actions, (0, MAX_ACTION_DIM - actions.shape[-1]))

    noise = policy.model.sample_noise(actions.shape, dev)
    t     = policy.model.sample_time(bs, dev)
    x_t   = t[:, None, None] * noise + (1.0 - t[:, None, None]) * actions
    u_t   = noise - actions

    # Set per-batch advantage signal for the patched embed_suffix
    if adv_values is not None:
        policy.model._train_advantage_values = adv_values
        if hasattr(policy.model, "_train_advantage_label"):
            del policy.model._train_advantage_label
    else:
        policy.model._train_advantage_label = adv_labels
        if hasattr(policy.model, "_train_advantage_values"):
            del policy.model._train_advantage_values

    prefix_embs, prefix_pad, _ = policy.model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state)
    suffix_embs, suffix_pad, _ = policy.model.embed_suffix(x_t, t)

    pad = torch.cat([prefix_pad, suffix_pad], dim=1)
    p_len, s_len = prefix_pad.shape[1], suffix_pad.shape[1]
    att = torch.cat([torch.zeros(bs, p_len, dtype=torch.bool, device=dev),
                     torch.ones(bs, s_len, dtype=torch.bool, device=dev)], dim=1)
    att_2d  = make_att_2d_masks(pad, att)
    pos_ids = torch.cumsum(pad, dim=1) - 1

    (_, suffix_out), _ = policy.model.vlm_with_expert.forward(
        attention_mask=att_2d, position_ids=pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, suffix_embs],
        use_cache=False, fill_kv_cache=False)
    suffix_out = suffix_out[:, -CHUNK_SIZE:].to(dtype=torch.float32)
    v_t = policy.model.action_out_proj(suffix_out)
    real_dim = policy.config.action_feature.shape[0]
    per = F.mse_loss(u_t[..., :real_dim], v_t[..., :real_dim], reduction="none")
    return per.flatten(1).mean(dim=1)


def load_fresh_policy_with_discrete_advantage():
    """Load BC baseline, attach discrete advantage embedding, install suffix patch, freeze VLM."""
    policy = SmolVLAPolicy.from_pretrained(BC_POLICY_REPO).to(device)
    expert_hidden = policy.model.vlm_with_expert.expert_hidden_size
    policy.model.advantage_token = AdvantageEmbedding(expert_hidden).to(device)
    cls = type(policy.model)
    if not hasattr(cls, "_orig_embed_suffix"):
        cls._orig_embed_suffix = cls.embed_suffix
    cls.embed_suffix = _patched_embed_suffix_discrete
    _freeze_vlm(policy)
    return policy


def load_fresh_policy_with_continuous_advantage():
    """Load BC baseline, attach continuous advantage MLP, install suffix patch, freeze VLM."""
    policy = SmolVLAPolicy.from_pretrained(BC_POLICY_REPO).to(device)
    expert_hidden = policy.model.vlm_with_expert.expert_hidden_size
    policy.model.advantage_token_continuous = ContinuousAdvantageEmbedding(expert_hidden).to(device)
    cls = type(policy.model)
    if not hasattr(cls, "_orig_embed_suffix"):
        cls._orig_embed_suffix = cls.embed_suffix
    cls.embed_suffix = _patched_embed_suffix_continuous
    _freeze_vlm(policy, extra_trainable=("advantage_token_continuous",))
    return policy


def _freeze_vlm(policy, extra_trainable=()):
    TRAINABLE_KEYWORDS = (
        "advantage_token",
        "action_in_proj", "action_out_proj",
        "action_time_mlp", "state_proj", "lm_expert",
    ) + extra_trainable
    for name, p in policy.named_parameters():
        p.requires_grad = any(k in name for k in TRAINABLE_KEYWORDS)
    trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in policy.parameters())
    print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M")


def run_training(policy, dataloader, run_name, save_dir, steps=1500, lr=1e-4,
                 use_continuous=False, log_every=50):
    """Generic training loop. Returns loss history list of (step, total, pos, neg)."""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    tokenizer  = policy.model.vlm_with_expert.processor.tokenizer
    CHUNK_SIZE = policy.config.chunk_size
    MAX_ACTION_DIM = policy.config.max_action_dim

    adv_key = "advantage_token_continuous" if use_continuous else "advantage_token"
    optimizer = torch.optim.AdamW(
        [{"params": [p for n, p in policy.named_parameters()
                     if adv_key in n and p.requires_grad], "lr": lr * 10},
         {"params": [p for n, p in policy.named_parameters()
                     if adv_key not in n and p.requires_grad], "lr": lr}],
        weight_decay=1e-10, betas=(0.9, 0.95))

    wb_run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY, mode=WANDB_MODE,
        name=run_name,
        config={"method": run_name, "steps": steps, "lr": lr,
                "batch": dataloader.batch_size, "continuous": use_continuous},
        reinit=True)

    policy.train()
    history = []
    running_tot = running_pos = running_neg = 0.0
    n_pos_b = n_neg_b = 0
    step = 0
    t0 = time.time()
    pbar = tqdm(total=steps, desc=run_name, dynamic_ncols=True)

    while step < steps:
        for batch in dataloader:
            if step >= steps:
                break
            batch = batch_to_device(batch, device)

            if use_continuous:
                adv_values = batch["value_score"].to(device).float()
                per_sample = recap_loss(policy, batch, tokenizer, CHUNK_SIZE, MAX_ACTION_DIM,
                                       adv_values=adv_values)
                pos_m = (adv_values > 0.5)
                neg_m = (adv_values <= 0.5)
            else:
                adv = batch["advantage_label"].to(device).long()
                per_sample = recap_loss(policy, batch, tokenizer, CHUNK_SIZE, MAX_ACTION_DIM,
                                       adv_labels=adv)
                pos_m = (adv == 1)
                neg_m = (adv == 0)

            loss = per_sample.mean()
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
            optimizer.step()

            with torch.no_grad():
                if pos_m.any(): running_pos += per_sample[pos_m].mean().item(); n_pos_b += 1
                if neg_m.any(): running_neg += per_sample[neg_m].mean().item(); n_neg_b += 1
                running_tot += loss.item()

            step += 1
            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item():.3f}"})

            if step % log_every == 0:
                p = running_pos / max(n_pos_b, 1)
                n = running_neg / max(n_neg_b, 1)
                t = running_tot / log_every
                d = p - n
                tqdm.write(f"[{run_name} step {step:5d}] loss={t:.4f} pos={p:.4f} neg={n:.4f} Δ={d:+.4f} elapsed={(time.time()-t0)/60:.1f}m")
                history.append((step, t, p, n))
                wb_run.log({"step": step, "loss/total": t, "loss/pos": p, "loss/neg": n, "loss/delta": d})
                running_tot = running_pos = running_neg = 0.0
                n_pos_b = n_neg_b = 0

    pbar.close()
    policy.save_pretrained(save_dir)
    with open(save_dir / "loss_history.json", "w") as f:
        json.dump(history, f)
    wb_run.summary["final_delta"] = history[-1][2] - history[-1][3] if history else 0
    wb_run.finish()
    print(f"  Saved to {save_dir}")
    return history


print("Training infrastructure ready.")

Training infrastructure ready.


## Step 9 — Eval harness (same monkey-patched rollout as recap_final)

In [10]:
from copy import deepcopy
from tqdm import trange
import lerobot.scripts.lerobot_eval as _le
from lerobot.envs.utils import preprocess_observation, add_envs_task, check_env_attributes_and_types
from lerobot.utils.constants import ACTION as _ACTION, OBS_STR
from lerobot.utils.utils import inside_slurm
from lerobot.envs.configs import LiberoEnv as LiberoEnvCfg
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.factory import make_pre_post_processors
from lerobot.scripts.lerobot_eval import eval_policy


def _patched_rollout(env, policy, env_preprocessor, env_postprocessor, preprocessor, postprocessor,
                     seeds=None, return_observations=False, render_callback=None):
    policy.reset()
    observation, info = env.reset(seed=seeds)
    if render_callback is not None: render_callback(env)
    all_actions, all_rewards, all_successes, all_dones = [], [], [], []
    step = 0
    done = np.array([False] * env.num_envs)
    max_steps = env.call("_max_episode_steps")[0]
    progbar = trange(max_steps, desc=f"rollout <=({max_steps} steps)", disable=inside_slurm(), leave=False)
    check_env_attributes_and_types(env)
    while not np.all(done) and step < max_steps:
        observation = preprocess_observation(observation)
        observation = add_envs_task(env, observation)
        observation = env_preprocessor(observation)
        observation = preprocessor(observation)
        with torch.inference_mode():
            action = policy.select_action(observation)
        action = postprocessor(action)
        action_transition = {_ACTION: action}
        action_transition = env_postprocessor(action_transition)
        action = action_transition[_ACTION]
        action_numpy = action.to("cpu").numpy()
        observation, reward, terminated, truncated, info = env.step(action_numpy)
        if render_callback is not None: render_callback(env)
        successes = info["final_info"]["is_success"].tolist() if "final_info" in info else [False] * env.num_envs
        done = terminated | truncated | done
        if step + 1 == max_steps: done = np.ones_like(done, dtype=bool)
        all_actions.append(torch.from_numpy(action_numpy))
        all_rewards.append(torch.from_numpy(reward))
        all_dones.append(torch.from_numpy(done))
        all_successes.append(torch.tensor(successes))
        step += 1; progbar.update()
    ret = {_ACTION: torch.stack(all_actions, dim=1), "reward": torch.stack(all_rewards, dim=1),
           "success": torch.stack(all_successes, dim=1), "done": torch.stack(all_dones, dim=1)}
    if hasattr(policy, "use_original_modules"): policy.use_original_modules()
    return ret

_le.rollout = _patched_rollout

# Build env preprocessors (policy-config-independent parts)
_tmp_policy = SmolVLAPolicy.from_pretrained(BC_POLICY_REPO)
_env_cfg = LiberoEnvCfg(task=LIBERO_SUITE, task_ids=[0])
env_preprocessor, env_postprocessor = make_env_pre_post_processors(env_cfg=_env_cfg, policy_cfg=_tmp_policy.config)
del _tmp_policy
torch.cuda.empty_cache()

print("Eval harness ready.")

config.json: 0.00B [00:00, ?B/s]

Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.03G [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Eval harness ready.


---
# ABLATION 1: Advantage Label Granularity

| Condition | Label source | Embedding type |
|-----------|-------------|----------------|
| (a) Binary episode-level | **Baseline** (recap_final — not re-run) | Discrete 3-way |
| **(b) Value-head thresholded** | V(s) > 0.5 → A_pos, else A_neg | Discrete 3-way |
| **(c) Continuous value** | V(s) scalar → MLP | Continuous MLP |

**Hypothesis:** If (b) or (c) improve Δ over (a), per-frame credit
assignment matters and binary episode-level labels are a bottleneck.

### A1(b): Value-Head Thresholded Labels

Relabel frames using V(s) > 0.5 → A_pos, V(s) ≤ 0.5 → A_neg.
Early frames in failed episodes get labeled positive if the value head
thinks the policy was still on track. Late frames in success episodes
can get labeled negative if something looked shaky.

In [11]:
class ValueThresholdedDataset(Dataset):
    """Wraps RolloutFrameDataset but overrides advantage_label
    based on per-frame value score threshold."""
    def __init__(self, base_ds, threshold=0.5):
        self.base_ds = base_ds
        self.threshold = threshold
        self._relabeled = np.zeros(len(base_ds), dtype=np.int64)
        n_flipped = 0
        for i in range(len(base_ds)):
            gi = base_ds._global_idx[i]
            v = float(base_ds._value_scores[gi])
            new_label = ADV_POS if v > threshold else ADV_NEG
            old_label = int(base_ds._labels[i])
            if new_label != old_label:
                n_flipped += 1
            self._relabeled[i] = new_label
        n_pos = int((self._relabeled == ADV_POS).sum())
        n_neg = int((self._relabeled == ADV_NEG).sum())
        print(f"Value-thresholded relabeling (thresh={threshold}):")
        print(f"  pos={n_pos}  neg={n_neg}  flipped={n_flipped} from original episode labels")

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, i):
        item = self.base_ds[i]
        item["advantage_label"] = int(self._relabeled[i])
        return item


vt_ds = ValueThresholdedDataset(full_ds, threshold=0.5)

# Mix with expert (expert always A_pos)
mixed_vt = ConcatDataset([expert_ds, vt_ds])
n_expert = len(expert_ds)
n_vt_pos = int((vt_ds._relabeled == 1).sum())
n_vt_neg = int((vt_ds._relabeled == 0).sum())
total_pos = n_expert + n_vt_pos
total_neg = n_vt_neg

w_pos = 1.0 / max(total_pos, 1)
w_neg = 1.0 / max(total_neg, 1)
weights_vt = np.empty(len(mixed_vt), dtype=np.float64)
weights_vt[:n_expert] = w_pos
for i in range(len(vt_ds)):
    weights_vt[n_expert + i] = w_pos if vt_ds._relabeled[i] == 1 else w_neg
sampler_vt = WeightedRandomSampler(weights=weights_vt, num_samples=len(mixed_vt), replacement=True)

dl_vt = DataLoader(mixed_vt, batch_size=16, sampler=sampler_vt,
                   num_workers=4, pin_memory=True, drop_last=True,
                   collate_fn=collate, persistent_workers=True, prefetch_factor=2)

print(f"\nA1(b) dataset ready: expert={n_expert} vt_pos={n_vt_pos} vt_neg={n_vt_neg}")

Value-thresholded relabeling (thresh=0.5):
  pos=2624  neg=12838  flipped=5361 from original episode labels

A1(b) dataset ready: expert=52970 vt_pos=2624 vt_neg=12838


In [ ]:
print("="*60)
print("TRAINING A1(b): Value-head thresholded labels")
print("="*60)

import os
os.environ["WANDB_API_KEY"] = "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"
wandb.login()

policy_a1b = load_fresh_policy_with_discrete_advantage()
history_a1b = run_training(
    policy_a1b, dl_vt,
    run_name="A1b_value_thresholded",
    save_dir=ABLATION_ROOT / "A1b_value_thresholded",
    steps=1500, lr=1e-4,
    use_continuous=False,
)

del policy_a1b
torch.cuda.empty_cache()

### A1(c): Continuous Value Embedding

Replace the discrete 3-way embedding with a continuous scalar V(s)
projected through a small MLP into expert_hidden_size. The model sees
the full value spectrum rather than a binary discretization.

In [13]:
# For continuous: expert (value_score=1.0) + rollouts (value_score from value head)
# Uniform random sampling — no pos/neg weighting since labels are continuous
mixed_cont = ConcatDataset([expert_ds, full_ds])

dl_cont = DataLoader(mixed_cont, batch_size=16, shuffle=True,
                     num_workers=4, pin_memory=True, drop_last=True,
                     collate_fn=collate, persistent_workers=True, prefetch_factor=2)

print(f"A1(c) dataset ready: expert={len(expert_ds)} + rollout={len(full_ds)} = {len(mixed_cont)}")

A1(c) dataset ready: expert=52970 + rollout=15462 = 68432


In [ ]:
print("="*60)
print("TRAINING A1(c): Continuous value embedding")
print("="*60)

policy_a1c = load_fresh_policy_with_continuous_advantage()
history_a1c = run_training(
    policy_a1c, dl_cont,
    run_name="A1c_continuous_value",
    save_dir=ABLATION_ROOT / "A1c_continuous_value",
    steps=1500, lr=1e-4,
    use_continuous=True,
)

del policy_a1c
torch.cuda.empty_cache()

---
# ABLATION 2: Negative-Only vs Positive-Only Training

| Condition | Training data | Rationale |
|-----------|--------------|----------|
| (a) Both labels | **Baseline** (recap_final — not re-run) | Mixed pos + neg |
| **(b) Negative-only** | Only A_neg frames from rollouts, no expert | Learn what failure looks like |
| **(c) Positive-only** | Only A_pos rollout frames + expert | Self-distillation, sharpen around success |

**Hypothesis:** BC already knows success (from pretraining).
If neg-only outperforms, the model benefits most from learning what to avoid.
If pos-only outperforms, self-distillation is sufficient.

### A2(b): Negative-Only Training

Train only on A_neg frames from rollout dataset. No expert data, no
positive frames. The advantage token is always A_neg during training.
At inference, we set A_pos to steer the model away from failure patterns.

In [17]:
neg_ds = RolloutFrameDataset(ROLLOUT_DATASET_DIR, value_scores=value_scores, label_filter=0)
print(f"Neg-only dataset: {len(neg_ds)} frames (all A_neg)")

dl_neg = DataLoader(neg_ds, batch_size=16, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True,
                    collate_fn=collate, persistent_workers=True, prefetch_factor=2)

Rollout frames: 7587  (filter=0)
Neg-only dataset: 7587 frames (all A_neg)


In [ ]:
print("="*60)
print("TRAINING A2(b): Negative-only")
print("="*60)

policy_a2b = load_fresh_policy_with_discrete_advantage()
history_a2b = run_training(
    policy_a2b, dl_neg,
    run_name="A2b_neg_only",
    save_dir=ABLATION_ROOT / "A2b_neg_only",
    steps=1500, lr=1e-4,
    use_continuous=False,
)

del policy_a2b
torch.cuda.empty_cache()

### A2(c): Positive-Only Training (Self-Distillation)

Train only on A_pos frames: success rollouts + expert demos.
The model never sees failures — it sharpens around success trajectories.

In [ ]:
pos_ds = RolloutFrameDataset(ROLLOUT_DATASET_DIR, value_scores=value_scores, label_filter=1)
print(f"Pos-only rollout: {len(pos_ds)} frames")

mixed_pos = ConcatDataset([expert_ds, pos_ds])
dl_pos = DataLoader(mixed_pos, batch_size=16, shuffle=True,
                    num_workers=4, pin_memory=True, drop_last=True,
                    collate_fn=collate, persistent_workers=True, prefetch_factor=2)

print(f"A2(c) dataset ready: expert={len(expert_ds)} + pos_rollout={len(pos_ds)} = {len(mixed_pos)}")

In [ ]:
print("="*60)
print("TRAINING A2(c): Positive-only (self-distillation)")
print("="*60)

policy_a2c = load_fresh_policy_with_discrete_advantage()
history_a2c = run_training(
    policy_a2c, dl_pos,
    run_name="A2c_pos_only",
    save_dir=ABLATION_ROOT / "A2c_pos_only",
    steps=1500, lr=1e-4,
    use_continuous=False,
)

del policy_a2c
torch.cuda.empty_cache()

---
## Step 10 — Training Loss Comparison Plots

In [ ]:
import matplotlib.pyplot as plt

# Load baseline history from recap_final
baseline_hist_path = Path(DRIVE_ROOT) / "recap_adv_final" / "loss_history.json"
if baseline_hist_path.exists():
    with open(baseline_hist_path) as f:
        history_baseline = json.load(f)
else:
    history_baseline = None
    print("WARNING: No baseline history found at", baseline_hist_path)

all_histories = {
    "Baseline (binary ep-level)": history_baseline,
    "A1b: Value thresholded": history_a1b,
    "A1c: Continuous value": history_a1c,
    "A2b: Neg-only": history_a2b,
    "A2c: Pos-only": history_a2c,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Total loss
ax = axes[0]
for name, hist in all_histories.items():
    if hist is None: continue
    h = np.array(hist)
    ax.plot(h[:, 0], h[:, 1], label=name, alpha=0.8)
ax.set_xlabel("step"); ax.set_ylabel("total loss")
ax.set_title("Total Loss"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Plot 2: Pos loss
ax = axes[1]
for name, hist in all_histories.items():
    if hist is None: continue
    h = np.array(hist)
    ax.plot(h[:, 0], h[:, 2], label=name, alpha=0.8)
ax.set_xlabel("step"); ax.set_ylabel("pos loss")
ax.set_title("Positive-Frame Loss"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Plot 3: Delta (pos - neg)
ax = axes[2]
for name, hist in all_histories.items():
    if hist is None: continue
    h = np.array(hist)
    ax.plot(h[:, 0], h[:, 2] - h[:, 3], label=name, alpha=0.8)
ax.axhline(0, color="k", ls="--", alpha=0.3)
ax.set_xlabel("step"); ax.set_ylabel("Δ (pos - neg)")
ax.set_title("Loss Delta (lower = better conditioning)"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle("RECAP Ablation: Conditioning Signal Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ABLATION_ROOT / "ablation_loss_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {ABLATION_ROOT / 'ablation_loss_comparison.png'}")

## Step 11 — Eval: All 4 ablation runs on tasks 0, 5, 7

Discrete runs use A_pos at inference. Continuous uses V=1.0.
Compare success rate against BC baseline numbers.

In [ ]:
import safetensors.torch as st

BC_PER_TASK = {0: 80, 5: 20, 7: 40}  # from 5-ep BC baseline
EVAL_TASKS  = [0, 5, 7]
N_EPISODES  = 3

ablation_configs = [
    {"name": "A1b_value_thresholded", "ckpt": ABLATION_ROOT / "A1b_value_thresholded", "mode": "discrete"},
    {"name": "A1c_continuous_value",  "ckpt": ABLATION_ROOT / "A1c_continuous_value",  "mode": "continuous"},
    {"name": "A2b_neg_only",          "ckpt": ABLATION_ROOT / "A2b_neg_only",          "mode": "discrete"},
    {"name": "A2c_pos_only",          "ckpt": ABLATION_ROOT / "A2c_pos_only",          "mode": "discrete"},
]

results_table = []

for cfg in ablation_configs:
    print(f"\n{'='*60}")
    print(f"Evaluating: {cfg['name']}")
    print(f"{'='*60}")

    pol = SmolVLAPolicy.from_pretrained(cfg["ckpt"]).to(device)
    expert_hidden = pol.model.vlm_with_expert.expert_hidden_size

    # Load trained advantage weights from safetensors checkpoint
    ckpt_files = sorted(cfg["ckpt"].glob("*.safetensors"))
    adv_state = {}
    for cf in ckpt_files:
        sd = st.load_file(cf)
        for k, v in sd.items():
            if "advantage_token" in k:
                adv_state[k] = v

    if cfg["mode"] == "discrete":
        pol.model.advantage_token = AdvantageEmbedding(expert_hidden).to(device)
        # Load weights (keys like "model.advantage_token.embedding.weight")
        for k, v in adv_state.items():
            parts = k.split("advantage_token.")
            if len(parts) == 2:
                local_k = parts[-1]
                if local_k in pol.model.advantage_token.state_dict():
                    pol.model.advantage_token.state_dict()[local_k].copy_(v)
        cls = type(pol.model)
        if not hasattr(cls, "_orig_embed_suffix"):
            cls._orig_embed_suffix = cls.embed_suffix
        cls.embed_suffix = _patched_embed_suffix_discrete
        set_eval_advantage(pol, ADV_POS)
    else:
        pol.model.advantage_token_continuous = ContinuousAdvantageEmbedding(expert_hidden).to(device)
        for k, v in adv_state.items():
            parts = k.split("advantage_token_continuous.")
            if len(parts) == 2:
                local_k = parts[-1]
                if local_k in pol.model.advantage_token_continuous.state_dict():
                    pol.model.advantage_token_continuous.state_dict()[local_k].copy_(v)
        cls = type(pol.model)
        if not hasattr(cls, "_orig_embed_suffix"):
            cls._orig_embed_suffix = cls.embed_suffix
        cls.embed_suffix = _patched_embed_suffix_continuous
        set_eval_advantage_continuous(pol, 1.0)

    pol.eval()

    preprocessor, postprocessor = make_pre_post_processors(
        policy_cfg=pol.config, pretrained_path=cfg["ckpt"],
        preprocessor_overrides={"device_processor": {"device": str(pol.config.device)}})

    env_cfg = LiberoEnvCfg(task=LIBERO_SUITE, task_ids=EVAL_TASKS)
    envs = make_env(env_cfg, n_envs=1, use_async_envs=False)

    for task_id in EVAL_TASKS:
        env = envs[LIBERO_SUITE][task_id]
        result = eval_policy(
            env=env, policy=pol,
            env_preprocessor=env_preprocessor, env_postprocessor=env_postprocessor,
            preprocessor=preprocessor, postprocessor=postprocessor,
            n_episodes=N_EPISODES, max_episodes_rendered=0, videos_dir=None,
            return_episode_data=False, start_seed=95000 + 1000 * task_id)
        successes = [bool(ep["success"]) for ep in result["per_episode"]]
        sr = 100 * sum(successes) / len(successes)
        bc = BC_PER_TASK[task_id]
        print(f"  Task {task_id}: {cfg['name']}={sr:.0f}%  BC={bc}%  Δ={sr-bc:+.0f}%")
        results_table.append((cfg["name"], task_id, sr, bc))

    del pol, envs
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("EVALUATION COMPLETE")
print(f"{'='*60}")

## Step 12 — Results Summary

In [ ]:
import pandas as pd

rows = [{"Ablation": n, "Task": t, "SR%": s, "BC%": b, "Δ vs BC": s - b}
        for n, t, s, b in results_table]
df = pd.DataFrame(rows)

print("\n=== Full Results Table ===")
print(df.to_string(index=False))

print("\n=== Average SR by Ablation ===")
print(df.groupby("Ablation")[["SR%", "Δ vs BC"]].mean().to_string())

df.to_csv(ABLATION_ROOT / "ablation_results.csv", index=False)
print(f"\nSaved to {ABLATION_ROOT / 'ablation_results.csv'}")

In [ ]:
# Final training deltas
print("\n=== Final Training Deltas (pos_loss - neg_loss) ===")
for name, hist in all_histories.items():
    if hist is None:
        print(f"  {name}: (no data — run from recap_final)")
        continue
    h = np.array(hist)
    final_delta = h[-1, 2] - h[-1, 3]
    print(f"  {name}: Δ = {final_delta:+.4f}")

print("\nInterpretation:")
print("  Δ < 0: model fits positive frames better than negative (conditioning works)")
print("  Δ ≈ 0: advantage token has no effect")
print("  Δ > 0: model fits negative frames better (inverted — possible issue)")